# 04 - Pipeline

**Importação de bibliotecas**

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

**Leitura do CSV preparado (saída da fase 2)**

In [2]:
df = pd.read_csv("data/heart_preparado.csv")

**Separar `X` e `y`**

In [3]:
y = df["has_disease"] # variavel alvo

X = df.drop(columns=["has_disease", "id", "dataset"])

**Dividir treino e teste - 80% treino, 20% teste**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Definindo as colunas**

In [5]:
numeric_cols = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
    "ca"
]

categorical_cols = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "thal"
]

# Não confunda o tipo de dado com a natureza da variável: uma coluna pode ser armazenada como object ou bool e ainda representar uma categoria.

Apesar de serem booleanas, `fbs` e `exang` representam categorias, não quantidades. Logo, elas devem ser armazenadas no grupo de colunas categóricas.

## Criação de 2 mini-pipelines

**Importação de bibliotecas**

In [6]:
from sklearn.pipeline import Pipeline                               # organiza transformações em sequência
from sklearn.impute import SimpleImputer                            # preenche valores ausentes
from sklearn.preprocessing import StandardScaler, OneHotEncoder     # standardscaler padroniza variáveis numéricas, onehotencoder transforma categorias em colunas numéricas

**Pipeline numérico**

In [7]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

- A sequência vai ser: dados numéricos (age, trestbps, chol, thalch, oldpeak, ca) → simpleIMputer → preenche NaN com a mediana → standardScaler (padroniza as escalas) → `dados numéricos transformados`

**Pipeline categórico**

In [8]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

- A sequência vai ser: dados categóricos (sex, cp, fbs, restecg, exang, slope, thal) → simpleIMputer (preenche NaN com a categoria mais frequente) → ondeHotEncoder (transforma categorias em colunas binárias) → `dados categóricos transformados`

**Importação do ColumnTransformer**

In [9]:
from sklearn.compose import ColumnTransformer

**Juntar os mini-pipelines**

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

- `"num"`: nome que identifica a transformação.
- `numeric_pipeline`: sequência de tratamento numérico.
- `numeric_cols`: colunas nas quais essa sequência será aplicada.

O mesmo raciocínio vale para o grupo categórico.

**Unindo o pré-processamento com o modelo:**

In [11]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)) # limite maior de iterações para permitir que a regressão logística encontre a solução de otimização, reduzindo a possibilidade de não convergência com o limite padrão.
])

Minha previsão pessoal é que a regressão logística alcance uma acurácia superior ao baseline de 55,4%, pois agora o modelo utilizará as variáveis clínicas após o tratamento de valores ausentes, padronização e codificação das categorias. Ainda não consigo afirmar com certeza se alcançará minha expectativa inicial de 87–94%, pois isso vai depender da capacidade dessas variáveis de distinguir pacientes com e sem doença.

**Treinamento e avaliação**

In [12]:
# treinar o pipeline com os dados de treino
model_pipeline.fit(X_train, y_train)

# avaliar nos dados de teste
accuracy = model_pipeline.score(X_test, y_test)

print(f"Acurácia no teste (pipeline): {accuracy:.2%}")

Acurácia no teste (pipeline): 80.98%


## Baseline x Pipeline
A regressão logística alcançou 80,98% de acurácia no conjunto de teste, superando o baseline de 55,4%. Esse resultado indica que as variáveis clínicas possuem informações relevantes para diferenciar pacientes com e sem doença cardíaca.

Entretanto, a acurácia ficou abaixo da minha expectativa inicial de 87–94%. Isso mostra que, apesar de o modelo apresentar um desempenho inicial satisfatório, ainda existe espaço para melhorias.

| Métrica  | Baseline | Regressão Logística | Expectativa inicial |
| -------- | -------- | ------------------- | ------------------- |
| **Acurácia** | 55,40%   | **80,98%**          | 87–94%              |

O pipeline também permitiu organizar o pré-processamento e o treinamento em uma única estrutura, evitando vazamento de dados ao ajustar as transformações somente com o conjunto de treino.

Como próximo passo, será necessário aprofundar a avaliação utilizando outras métricas, como precisão, recall, F1-score e matriz de confusão, para compreender melhor os erros cometidos pelo modelo.

---


**Quando você chama fit(X_train) no pipeline, onde a mediana é calculada? E quando chama score(X_test), ela é recalculada?**

Quando chamamos fit(X_train), o SimpleImputer calcula a mediana de cada coluna numérica utilizando somente os dados de treinamento. Esses valores ficam armazenados para serem utilizados nas próximas transformações.

Quando chamamos score(X_test), a mediana não é recalculada. O pipeline utiliza as medianas que aprendeu durante o treinamento para preencher os valores ausentes no conjunto de teste.

Isso evita o vazamento de dados (data leakage), pois informações do conjunto de teste não são utilizadas para aprender os parâmetros do pré processamento. Dessa forma, a avaliação representa melhor o comportamento do modelo diante de dados novos.

| Etapa | O que acontece? |
| --- | --- |
| `fit(X_train)` | calcula e armazena as medianas com os dados de treino |
| `score(X_test)` | utiliza as medianas armazenadas, sem recalculá-las. |
